# YouTube collection — COVID-19 vaccine claims

Collect candidate YouTube videos discussing COVID-19 vaccines, with their metadata and statistics, to build a dataset for **downstream fact-checking** (the `rag.ipynb` notebook).

## Methodology & scope

This notebook **collects candidates** — it does *not* fetch transcripts and does *not* judge whether a video is true or false. Both are the job of later stages:

1. **This notebook (collection):** search COVID-vaccine-related videos via the official **YouTube Data API v3**, fetch metadata + statistics, and save a neutral dataset to `data/`.
2. **`transcribe.ipynb` (transcripts):** download each video's audio and transcribe it with Whisper.
3. **`rag.ipynb` (classification):** extract claims and check them against evidence to identify which contain misinformation.

Keeping collection and classification separate is what makes this a defensible research method: you gather a neutral candidate pool, you don't pre-judge or target creators. COVID-vaccine misinformation detection is a well-established research area; this is standard data-gathering for it.

### Why the official API (not HTML scraping)
- Scraping youtube.com violates YouTube's Terms of Service, breaks when the markup changes, and gets IPs blocked.
- The **Data API v3** is official, free (10,000 quota units/day), returns structured data, and is citable in a thesis methods section.

This notebook is the **final experiment run** — every cell is part of the pipeline. Verification scripts (link-parser checks, schema and merge-logic checks) live in `tests/`; see `tests/README.md`.

## Setup

Add a YouTube Data API key to `.env` (free — create one in [Google Cloud Console](https://console.cloud.google.com/) → enable "YouTube Data API v3" → create an API key):

```
YOUTUBE_API_KEY=...
```

> This key is distinct from `GEMINI_API_KEY` (Gemini) — separate Google products, separate keys.

In [ ]:
# Run once.
%pip install -q google-api-python-client pandas python-dotenv

## 1. Configuration

Loads the API key and sets collection parameters. Mind the **quota**: a `search.list` call costs **100 units**, so the free 10,000/day budget allows ~100 searches. Each search returns up to 50 results.

In [ ]:
from __future__ import annotations

import os
import json
import time
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

# Collection settings
MAX_RESULTS_PER_QUERY = 25   # 1-50; results per search query
RELEVANCE_LANGUAGE = "en"    # bias results to English; the dataset is English-only
PUBLISHED_AFTER = None       # e.g. "2020-01-01T00:00:00Z" to limit by date; None = all

OUTPUT_DIR = PROJECT_ROOT / "data" / "youtube"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not YOUTUBE_API_KEY:
    print("WARNING: YOUTUBE_API_KEY not set — add it to .env before running the collection cells.")
else:
    print("YouTube API key loaded.")
print(f"Output dir: {OUTPUT_DIR}")

## 2. Search queries

The search terms that define the candidate pool. These are **neutral topic queries** about COVID-19 vaccines — they cast a wide net; the fact-checking stage decides what's actually false. Edit this list to focus the dataset on the specific claims you're studying.

The dataset is **English-only**: queries are English and `RELEVANCE_LANGUAGE = "en"` biases results accordingly. `transcribe.ipynb` detects the spoken language and drops anything non-English that slips through, since search-result language is only a hint.

> Tip: searching for the *misinformation narratives themselves* (e.g. "vaccine microchip", "vaccine causes infertility") surfaces more candidate videos than neutral terms — but keep the framing as "topics people make claims about", not "true facts".

In [2]:
SEARCH_QUERIES: list[str] = [
    "covid vaccine side effects",
    "covid vaccine truth",
    "covid vaccine dangers",
    "covid vaccine mRNA risks",
    "covid vaccine microchip",       # known misinformation narrative
    "covid vaccine infertility",     # known misinformation narrative
]

print(f"{len(SEARCH_QUERIES)} search queries configured.")
for q in SEARCH_QUERIES:
    print("  -", q)

6 search queries configured.
  - covid vaccine side effects
  - covid vaccine truth
  - covid vaccine dangers
  - covid vaccine mRNA risks
  - covid vaccine microchip
  - covid vaccine infertility


## 3. YouTube Data API client and search

Defines the `Video` record and `search_videos()`. Each `search.list` call costs 100 quota units, so nothing here calls the API — section 5 runs the searches.

Note that `search.list` returns **truncated** descriptions (~140 chars). Section 8 backfills the full text.

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

_youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY) if YOUTUBE_API_KEY else None


@dataclass
class Video:
    video_id: str
    title: str
    description: str
    channel: str
    channel_id: str
    published_at: str
    query: str
    url: str = ""
    view_count: Optional[int] = None
    like_count: Optional[int] = None
    comment_count: Optional[int] = None

    def __post_init__(self):
        if not self.url:
            self.url = f"https://www.youtube.com/watch?v={self.video_id}"


def search_videos(query: str, max_results: int = MAX_RESULTS_PER_QUERY) -> list[Video]:
    """Return candidate videos for one query (snippet only; stats fetched separately)."""
    if _youtube is None:
        raise RuntimeError("YOUTUBE_API_KEY not set.")
    params = dict(q=query, part="snippet", type="video",
                  maxResults=max(1, min(50, max_results)))
    if RELEVANCE_LANGUAGE:
        params["relevanceLanguage"] = RELEVANCE_LANGUAGE
    if PUBLISHED_AFTER:
        params["publishedAfter"] = PUBLISHED_AFTER

    resp = _youtube.search().list(**params).execute()
    videos = []
    for item in resp.get("items", []):
        sn = item["snippet"]
        videos.append(Video(
            video_id=item["id"]["videoId"],
            title=sn.get("title", ""),
            description=sn.get("description", ""),
            channel=sn.get("channelTitle", ""),
            channel_id=sn.get("channelId", ""),
            published_at=sn.get("publishedAt", ""),
            query=query,
        ))
    return videos


print("search_videos() ready.")

## 4. Fetch statistics (views, likes, comments)

`search.list` doesn't include stats. `videos.list` does, and is cheap (1 unit per call, up to 50 IDs per call).

In [4]:
def attach_statistics(videos: list[Video]) -> None:
    """Fill view/like/comment counts in place, batching 50 IDs per call."""
    if _youtube is None or not videos:
        return
    by_id = {v.video_id: v for v in videos}
    ids = list(by_id)
    for i in range(0, len(ids), 50):
        batch = ids[i:i + 50]
        resp = _youtube.videos().list(part="statistics", id=",".join(batch)).execute()
        for item in resp.get("items", []):
            st = item.get("statistics", {})
            v = by_id.get(item["id"])
            if v:
                v.view_count = int(st["viewCount"]) if "viewCount" in st else None
                v.like_count = int(st["likeCount"]) if "likeCount" in st else None
                v.comment_count = int(st["commentCount"]) if "commentCount" in st else None


print("attach_statistics() ready.")

attach_statistics() ready.


## 5. Run the collection

Searches every query, dedupes by video ID, and attaches statistics. A small `sleep` between searches keeps the request rate polite.

In [5]:
collected: dict[str, Video] = {}

if _youtube is None:
    print("No API key — set YOUTUBE_API_KEY in .env and re-run from section 1.")
else:
    # 1) Search all queries, dedupe by video_id (a video can match several queries).
    for q in SEARCH_QUERIES:
        try:
            for v in search_videos(q):
                if v.video_id not in collected:
                    collected[v.video_id] = v
            print(f"  searched: {q!r:40}  total unique so far: {len(collected)}")
        except HttpError as e:
            print(f"  SEARCH ERROR for {q!r}: {e}")
        time.sleep(0.3)

    videos = list(collected.values())

    # 2) Stats (cheap, batched).
    attach_statistics(videos)

    print(f"\nDone. {len(videos)} unique videos with metadata and statistics.")

  searched: 'covid vaccine side effects'              total unique so far: 25
  searched: 'covid vaccine truth'                     total unique so far: 47
  searched: 'covid vaccine dangers'                   total unique so far: 58
  searched: 'covid vaccine mRNA risks'                total unique so far: 76
  searched: 'covid vaccine microchip'                 total unique so far: 101
  searched: 'covid vaccine infertility'               total unique so far: 126

Done. 126 unique videos with metadata and statistics.


## 6. Save the dataset

Writes both CSV (for inspection / spreadsheets) and JSON to `data/youtube/`. The JSON is what the fact-checking notebook reads to pick up its videos.

In [6]:
import pandas as pd

if collected:
    rows = [asdict(v) for v in collected.values()]
    df = pd.DataFrame(rows)

    stamp = time.strftime("%Y%m%d_%H%M%S")
    csv_path = OUTPUT_DIR / f"covid_vaccine_videos_{stamp}.csv"
    json_path = OUTPUT_DIR / f"covid_vaccine_videos_{stamp}.json"

    df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(df)} videos:")
    print(f"  {csv_path}")
    print(f"  {json_path}")
    print(f"\nUnique channels: {df['channel'].nunique()}")
    print(f"Videos per query:")
    print(df["query"].value_counts().to_string())
else:
    print("Nothing collected yet — run section 5 first.")

Saved 126 videos:
  /Users/heloysapelizon/Desktop/TCC/data/youtube/covid_vaccine_videos_20260905_222751.csv
  /Users/heloysapelizon/Desktop/TCC/data/youtube/covid_vaccine_videos_20260905_222751.json

Unique channels: 110
Videos per query:
query
covid vaccine side effects    25
covid vaccine microchip       25
covid vaccine infertility     25
covid vaccine truth           22
covid vaccine mRNA risks      18
covid vaccine dangers         11


## 7. Collect from a list of links

Search casts a wide net. Sometimes you instead have a **specific list of videos** — e.g. the [Disinformation Dozen](https://counterhate.com/research/the-disinformation-dozen/) accounts — and want exactly those in the dataset.

Put one link per line in a `.txt` file inside `data/youtube/`. Blank lines and `#` comments are ignored, so the list can be annotated:

```
# Disinformation Dozen — sample videos
https://www.youtube.com/watch?v=21hkEfN3UmA
https://youtu.be/33vf9wJv_lw          # short form works
https://www.youtube.com/shorts/TDLu5BJzaAE
pQl_p4GtCOo                            # a bare 11-char ID works too
```

The cell calls `videos.list` to fetch **real** titles, channels, descriptions and statistics, so records are identical in shape to the search path and drop straight into `transcribe.ipynb`. The `query` field records provenance as `link_list:<filename>` — that's how you tell curated videos from searched ones later.

Cost is negligible: 1 quota unit per 50 videos, versus 100 for a single search.

In [ ]:
import re

# The .txt file holding one video link (or bare ID) per line.
LINKS_FILE = OUTPUT_DIR / "disinformation_dozen.txt"

# Matches the ID in watch?v=, youtu.be/, /shorts/, /embed/, or a bare 11-char ID.
_ID_PATTERNS = [
    re.compile(r"(?:youtube\.com/watch\?(?:.*&)?v=)([A-Za-z0-9_-]{11})"),
    re.compile(r"(?:youtu\.be/)([A-Za-z0-9_-]{11})"),
    re.compile(r"(?:youtube\.com/(?:shorts|embed|live)/)([A-Za-z0-9_-]{11})"),
    re.compile(r"^([A-Za-z0-9_-]{11})$"),
]


def parse_links_file(path: Path) -> tuple[list[str], list[str]]:
    """Read a links .txt and return (video_ids, unparseable_lines).

    Ignores blank lines and #-comments; strips trailing inline comments.
    Deduplicates while preserving file order.
    """
    if not path.exists():
        raise FileNotFoundError(f"No links file at {path}")

    ids: list[str] = []
    seen: set[str] = set()
    bad: list[str] = []

    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.split("#", 1)[0].strip()   # drop comments, inline included
        if not line:
            continue
        for pat in _ID_PATTERNS:
            m = pat.search(line)
            if m:
                vid = m.group(1)
                if vid not in seen:
                    seen.add(vid)
                    ids.append(vid)
                break
        else:
            bad.append(raw.strip())
    return ids, bad


def fetch_videos_by_id(video_ids: list[str], source: str) -> list[Video]:
    """Fetch full metadata + statistics for explicit video IDs.

    One videos.list call per 50 IDs (1 quota unit each). IDs the API does not
    return — deleted, private, or mistyped — are reported as missing.
    """
    if _youtube is None:
        raise RuntimeError("YOUTUBE_API_KEY not set.")
    if not video_ids:
        return []

    out: list[Video] = []
    found: set[str] = set()

    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i + 50]
        resp = _youtube.videos().list(
            part="snippet,statistics", id=",".join(batch)
        ).execute()
        for item in resp.get("items", []):
            sn = item["snippet"]
            st = item.get("statistics", {})
            found.add(item["id"])
            out.append(Video(
                video_id=item["id"],
                title=sn.get("title", ""),
                description=sn.get("description", ""),
                channel=sn.get("channelTitle", ""),
                channel_id=sn.get("channelId", ""),
                published_at=sn.get("publishedAt", ""),
                query=source,
                view_count=int(st["viewCount"]) if "viewCount" in st else None,
                like_count=int(st["likeCount"]) if "likeCount" in st else None,
                comment_count=int(st["commentCount"]) if "commentCount" in st else None,
            ))
        time.sleep(0.2)

    missing = [v for v in video_ids if v not in found]
    if missing:
        print(f"  {len(missing)} ID(s) returned nothing (deleted/private/typo): {missing}")
    return out


link_videos: list[Video] = []

if _youtube is None:
    print("No API key — set YOUTUBE_API_KEY in .env and re-run from section 1.")
elif not LINKS_FILE.exists():
    print(f"No links file yet. Create {LINKS_FILE.name} in {OUTPUT_DIR} "
          "with one link per line, then re-run this cell.")
else:
    ids, bad = parse_links_file(LINKS_FILE)
    print(f"{LINKS_FILE.name}: {len(ids)} unique video ID(s) parsed.")
    if bad:
        print(f"  {len(bad)} line(s) not recognised as links or IDs:")
        for b in bad[:10]:
            print(f"    {b[:70]}")

    if ids:
        link_videos = fetch_videos_by_id(ids, source=f"link_list:{LINKS_FILE.stem}")
        print(f"\nFetched metadata for {len(link_videos)}/{len(ids)} video(s):\n")
        for v in link_videos:
            views = f"{v.view_count:,}" if v.view_count is not None else "?"
            print(f"  {v.video_id}  {views:>12} views  {v.channel[:24]:24}  {v.title[:44]}")
    else:
        print("  File is empty — add links and re-run.")

### 7b. Save the link-based set

Writes the curated videos to their own `<listname>.json` / `.csv`, named after the links file.

Set `MERGE_INTO_MAIN = True` to also fold them into a combined dataset alongside the searched videos. The merge dedupes by `video_id`, and search results win on collision so a video's original `query` provenance is preserved.

In [ ]:
import pandas as pd   # also imported in section 6; repeated so 7b can run standalone

MERGE_INTO_MAIN = False   # True also writes a combined searched + curated dataset

if not link_videos:
    print("Nothing to save — run section 7 first.")
else:
    link_rows = [asdict(v) for v in link_videos]
    link_df = pd.DataFrame(link_rows)

    stem = LINKS_FILE.stem
    link_json = OUTPUT_DIR / f"{stem}.json"
    link_csv = OUTPUT_DIR / f"{stem}.csv"

    link_df.to_csv(link_csv, index=False)
    with open(link_json, "w", encoding="utf-8") as f:
        json.dump(link_rows, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(link_df)} curated videos:")
    print(f"  {link_json}")
    print(f"  {link_csv}")
    print(f"\nChannels in this list: {link_df['channel'].nunique()}")

    if MERGE_INTO_MAIN:
        # `collected` comes from section 5; empty if only section 7 was run.
        searched = [asdict(v) for v in collected.values()] if "collected" in dir() and collected else []

        # Searched videos first so they win the dedupe and keep their query value.
        merged: dict[str, dict] = {r["video_id"]: r for r in searched}
        added = 0
        for r in link_rows:
            if r["video_id"] not in merged:
                merged[r["video_id"]] = r
                added += 1

        stamp = time.strftime("%Y%m%d_%H%M%S")
        merged_rows = list(merged.values())
        merged_json = OUTPUT_DIR / f"covid_vaccine_videos_{stamp}_combined.json"
        merged_csv = OUTPUT_DIR / f"covid_vaccine_videos_{stamp}_combined.csv"

        pd.DataFrame(merged_rows).to_csv(merged_csv, index=False)
        with open(merged_json, "w", encoding="utf-8") as f:
            json.dump(merged_rows, f, ensure_ascii=False, indent=2)

        print(f"\nCombined dataset: {len(merged_rows)} videos "
              f"({len(searched)} searched + {added} new from the link list)")
        print(f"  {merged_json}")
        print(f"  {merged_csv}")

## 8. Backfill full descriptions (in place)

`search.list` returns **truncated** descriptions — roughly 140 characters, usually ending in `...`. `videos.list` returns the full text. Anything collected through the search path therefore has descriptions cut short, while section 7's link-based records already hold the complete text.

This cell repairs every `*.json` in `data/youtube/` in place:

1. Collect the union of `video_id`s across all dataset files.
2. Fetch full snippets once per video, batching 50 per `videos.list` call — a video appearing in several files is fetched once.
3. Rewrite each file with full `description` values, leaving every other field and the record order untouched.

Cost is trivial — 1 quota unit per 50 videos (~4 units for this whole corpus) against a 10,000/day budget.

**Safety.** These files are overwritten, so the cell writes a `.bak` copy of each first (set `MAKE_BACKUPS = False` to skip). Set `DRY_RUN = True` to preview the changes without writing anything — worth doing on the first pass.

Videos that have since been deleted or made private return nothing from the API; their existing truncated description is kept as-is and they are listed at the end.

In [ ]:
import shutil

DRY_RUN = False        # True = report what would change, write nothing
MAKE_BACKUPS = True    # write <name>.json.bak before overwriting

# Files to skip: not video datasets.
SKIP_FILES = {"transcripts_checkpoint.json"}


def load_dataset_files(directory: Path) -> tuple[dict[Path, list], list[tuple[Path, str]]]:
    """Return ({path: records}, [(path, reason)]) for every video dataset JSON."""
    good: dict[Path, list] = {}
    skipped: list[tuple[Path, str]] = []

    for path in sorted(directory.glob("*.json")):
        if path.name in SKIP_FILES or path.name.endswith(".bak"):
            skipped.append((path, "skipped by name"))
            continue
        try:
            with open(path, encoding="utf-8") as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            skipped.append((path, f"invalid JSON: {e}"))
            continue
        if not isinstance(data, list) or not data or not isinstance(data[0], dict):
            skipped.append((path, "not a list of records"))
            continue
        if "video_id" not in data[0]:
            skipped.append((path, "no video_id field"))
            continue
        good[path] = data
    return good, skipped


def fetch_full_snippets(video_ids: list[str]) -> dict[str, dict]:
    """Return {video_id: snippet} with untruncated descriptions, 50 IDs per call."""
    if _youtube is None:
        raise RuntimeError("YOUTUBE_API_KEY not set.")
    out: dict[str, dict] = {}
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i + 50]
        resp = _youtube.videos().list(part="snippet", id=",".join(batch)).execute()
        for item in resp.get("items", []):
            out[item["id"]] = item["snippet"]
        print(f"  fetched {min(i + 50, len(video_ids))}/{len(video_ids)}")
        time.sleep(0.2)
    return out


if _youtube is None:
    print("No API key — set YOUTUBE_API_KEY in .env and re-run from section 1.")
else:
    datasets, skipped = load_dataset_files(OUTPUT_DIR)

    print(f"Dataset files found: {len(datasets)}")
    for p, recs in datasets.items():
        trunc = sum(1 for r in recs if (r.get("description") or "").rstrip().endswith("..."))
        print(f"  {p.name:52} {len(recs):4d} records, {trunc:4d} truncated")
    for p, reason in skipped:
        print(f"  {p.name:52} SKIPPED — {reason}")

    all_ids = sorted({r["video_id"] for recs in datasets.values() for r in recs})
    print(f"\nUnique videos across all files: {len(all_ids)}")
    print(f"Quota cost: {(len(all_ids) + 49) // 50} unit(s)\n")

    snippets = fetch_full_snippets(all_ids)
    print(f"\nAPI returned {len(snippets)}/{len(all_ids)} videos.")

    missing_ids = [v for v in all_ids if v not in snippets]
    total_updated = 0

    print()
    for path, records in datasets.items():
        updated = 0
        grew_chars = 0
        for r in records:
            sn = snippets.get(r["video_id"])
            if not sn:
                continue
            full = sn.get("description", "")
            old = r.get("description") or ""
            if full != old:
                r["description"] = full
                updated += 1
                grew_chars += len(full) - len(old)

        verb = "would update" if DRY_RUN else "updated"
        print(f"  {path.name:52} {verb} {updated:4d} record(s), "
              f"+{grew_chars:,} chars")

        if updated and not DRY_RUN:
            if MAKE_BACKUPS:
                shutil.copy2(path, path.with_suffix(".json.bak"))
            # Write to a temp file then replace, so an interrupt cannot truncate
            # the original.
            tmp = path.with_suffix(".json.tmp")
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump(records, f, ensure_ascii=False, indent=2)
            tmp.replace(path)
        total_updated += updated

    print(f"\n{'DRY RUN — nothing written. ' if DRY_RUN else ''}"
          f"{total_updated} record(s) across {len(datasets)} file(s).")
    if MAKE_BACKUPS and not DRY_RUN and total_updated:
        print("Backups written alongside each file as *.json.bak")
    if missing_ids:
        print(f"\n{len(missing_ids)} video(s) unavailable (deleted/private) — "
              f"description left unchanged:")
        for v in missing_ids:
            print(f"  {v}")